# 🐷 ConvNeXt-DANN (UDA + Focal) – Inference & Submission

Generates submission CSV from a trained **ConvNeXt-DANN** checkpoint (384x384, UDA, Focal Loss).  
Das DANN-Modell nutzt kamera-invariante Features → bessere Generalisierung auf **unbekannte Kamera-Perspektiven**.  
Bei der Inference wird nur der **Posture-Head** verwendet, der Domain-Head wird ignoriert.  
Supports **TTA** (4 augmented views) and **multi-checkpoint ensembling**.

In [ ]:
import gc
import torch
import os

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'max_split_size_mb:128'
print("🧹 GPU Memory bereinigt.")

## ⚙️ Configuration

In [17]:
TAG = "T2"

DATA_ROOT   = "/datasets/multi-view-pig-posture-recognition"
TEST_CSV    = f"{DATA_ROOT}/test.csv"
IMG_DIR     = f"{DATA_ROOT}/test_images"

CKPT_PATHS  = [f"runs/convnext_dann_uda4_{TAG.lower()}/best_model.pth"]
OUTPUT_FILE = f"{TAG}_convnext_dann_uda_submission.csv"

IMG_SIZE     = 288
BATCH_SIZE   = 24
NUM_WORKERS  = 8
USE_TTA      = True
PAD_RATIO    = 0.25

NUM_CLASSES  = 5
CLASS_NAMES  = ["Lateral_lying_left", "Lateral_lying_right",
                "Sitting", "Standing", "Sternal_lying"]

print(f"Tag: {TAG}  |  Output: {OUTPUT_FILE}  |  TTA: {USE_TTA}  |  ImgSize: {IMG_SIZE}")
print(f"Checkpoints: {CKPT_PATHS}")

Tag: T2  |  Output: T2_convnext_dann_uda_submission.csv  |  TTA: True  |  ImgSize: 288
Checkpoints: ['runs/convnext_dann_uda4_t2/best_model.pth']


## 📚 Imports & SSL Fix

In [18]:
import ast
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm
import ssl
import urllib3
import requests

# --- SSL Fix für Zertifikat-Fehler ---
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
ssl._create_default_https_context = ssl._create_unverified_context
old_request = requests.Session.request
def new_request(self, method, url, **kwargs):
    kwargs['verify'] = False
    return old_request(self, method, url, **kwargs)
requests.Session.request = new_request
# --------------------------------------

import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast
import torchvision.transforms as T
import timm

import warnings
warnings.filterwarnings("ignore")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

Device: cuda


## 🧠 Model Architecture (must match training)

In [19]:
class GradientReversalFunction(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, lambda_):
        ctx.lambda_ = lambda_
        return x.clone()

    @staticmethod
    def backward(ctx, grad_output):
        return -ctx.lambda_ * grad_output, None


class GradientReversalLayer(nn.Module):
    def __init__(self, lambda_=1.0):
        super().__init__()
        self.lambda_ = lambda_

    def forward(self, x):
        return GradientReversalFunction.apply(x, self.lambda_)


class ConvNeXtDANN(nn.Module):
    """ConvNeXt with Domain Adversarial Training."""
    def __init__(self, model_name, num_classes, num_domains, drop_rate=0.3):
        super().__init__()
        self.features = timm.create_model(model_name, pretrained=False, num_classes=0)
        feat_dim = self.features.num_features
        
        self.posture_head = nn.Sequential(
            nn.LayerNorm(feat_dim),
            nn.Dropout(drop_rate),
            nn.Linear(feat_dim, 512),
            nn.GELU(),
            nn.Dropout(drop_rate),
            nn.Linear(512, num_classes)
        )
        
        self.grl = GradientReversalLayer()
        self.domain_head = nn.Sequential(
            nn.LayerNorm(feat_dim),
            nn.Dropout(drop_rate),
            nn.Linear(feat_dim, 512),
            nn.GELU(),
            nn.Dropout(drop_rate),
            nn.Linear(512, 256),
            nn.GELU(),
            nn.Dropout(drop_rate),
            nn.Linear(256, num_domains)
        )
    
    def forward(self, x, lambda_=1.0):
        features = self.features(x)
        posture_logits = self.posture_head(features)
        self.grl.lambda_ = lambda_
        domain_logits = self.domain_head(self.grl(features))
        return posture_logits, domain_logits
    
    def predict_posture(self, x):
        """Inference: only posture head."""
        features = self.features(x)
        return self.posture_head(features)

print("Model architecture defined.")

Model architecture defined.


## 🗂️ Test Dataset & TTA Transforms

In [20]:
class PigTestDataset(Dataset):
    def __init__(self, df, img_dir, transform=None, pad_ratio=0.25):
        self.df        = df.reset_index(drop=True)
        self.img_dir   = img_dir
        self.transform = transform
        self.pad_ratio = pad_ratio

    def __len__(self): return len(self.df)

    def _crop(self, img, bbox):
        W, H = img.size
        x, y, w, h = [float(v) for v in ast.literal_eval(bbox)]
        px, py = w * self.pad_ratio, h * self.pad_ratio
        x1 = max(0, int(x - px));  y1 = max(0, int(y - py))
        x2 = min(W, int(x+w+px));  y2 = min(H, int(y+h+py))
        return img.crop((x1, y1, x2, y2))

    def __getitem__(self, idx):
        row  = self.df.iloc[idx]
        img  = Image.open(os.path.join(self.img_dir, row["image_id"])).convert("RGB")
        crop = self._crop(img, row["bbox"])
        if self.transform: crop = self.transform(crop)
        return crop, row["row_id"]

S = IMG_SIZE
NORM = [[0.485,0.456,0.406],[0.229,0.224,0.225]]

TTA_TRANSFORMS = [
    T.Compose([T.Resize((S, S)), T.ToTensor(), T.Normalize(*NORM)]),
    T.Compose([T.Resize((S, S)), T.RandomHorizontalFlip(p=1.0), T.ToTensor(), T.Normalize(*NORM)]),
    T.Compose([T.Resize((S+32, S+32)), T.CenterCrop(S), T.ToTensor(), T.Normalize(*NORM)]),
    T.Compose([T.Resize((S+32, S+32)), T.CenterCrop(S), T.RandomHorizontalFlip(p=1.0), T.ToTensor(), T.Normalize(*NORM)]),
]
print(f"TTA: {len(TTA_TRANSFORMS)} Varianten bei {S}x{S} Auflösung.")

TTA: 4 Varianten bei 288x288 Auflösung.


## 🔮 Run Inference (Posture-Head Only)

In [21]:
test_df = pd.read_csv(TEST_CSV)
print(f"Test instances: {len(test_df)}")

@torch.no_grad()
def predict_tta(model, df, img_dir, transforms):
    """TTA inference using ONLY the posture head."""
    all_probs = []
    for i, tf in enumerate(transforms):
        ds     = PigTestDataset(df, img_dir, transform=tf, pad_ratio=PAD_RATIO)
        loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False,
                            num_workers=NUM_WORKERS, pin_memory=True)
        probs = []
        for imgs, _ in tqdm(loader, desc=f"  TTA {i+1}/{len(transforms)}", leave=False):
            with autocast():
                logits = model.predict_posture(imgs.to(DEVICE))
            probs.append(F.softmax(logits, dim=1).cpu().numpy())
        all_probs.append(np.vstack(probs))
    return np.mean(all_probs, axis=0)

transforms = TTA_TRANSFORMS if USE_TTA else [TTA_TRANSFORMS[0]]
ensemble_probs = []

for path in CKPT_PATHS:
    print(f"\nLoading: {path}")
    ckpt = torch.load(path, map_location="cpu")
    
    name = ckpt.get("model_name", "convnext_base.fb_in22k_ft_in1k_384")
    num_classes = ckpt.get("num_classes", NUM_CLASSES)
    num_domains = ckpt.get("num_domains", 22)  # fallback
    
    print(f"  Model: {name}  |  Architecture: {ckpt.get('architecture', 'ConvNeXtDANN')}")
    print(f"  Val F1: {ckpt.get('val_f1', 0):.4f}  |  Domain Acc: {ckpt.get('val_domain_acc', 'N/A')}")
    print(f"  Classes: {num_classes}  |  Domains: {num_domains}  |  ImgSize: {ckpt.get('img_size', IMG_SIZE)}")
    
    model = ConvNeXtDANN(name, num_classes, num_domains)
    model.load_state_dict(ckpt["model"])
    model.to(DEVICE).eval()
    
    ensemble_probs.append(predict_tta(model, test_df, IMG_DIR, transforms))
    del model; torch.cuda.empty_cache()

predictions = np.mean(ensemble_probs, axis=0).argmax(axis=1)
print(f"\n✓ Done – {len(predictions)} predictions")

Test instances: 11708

Loading: runs/convnext_dann_uda4_t2/best_model.pth
  Model: convnext_base.fb_in22k_ft_in1k_384  |  Architecture: ConvNeXtDANN_UDA_Focal
  Val F1: 0.9342  |  Domain Acc: 0.22473347547974415
  Classes: 5  |  Domains: 40  |  ImgSize: 288



✓ Done – 11708 predictions


## 📄 Save Submission

In [23]:
submission = pd.DataFrame({"row_id": test_df["row_id"].values, "class_id": predictions.astype(int)})
submission.to_csv(OUTPUT_FILE, index=False)
print(f"✓ Saved → {OUTPUT_FILE}  ({len(submission)} rows)")

assert list(submission.columns) == ["row_id", "class_id"]
assert set(submission["class_id"].unique()).issubset(set(range(5)))
assert len(submission) == len(test_df)
print("✓ Sanity checks passed")

print("\nPredicted distribution:")
counts = submission["class_id"].value_counts().sort_index()
for c in range(NUM_CLASSES):
    cnt = counts.get(c, 0)
    print(f"  {c} - {CLASS_NAMES[c]:<22} {cnt:>5}  ({100*cnt/len(submission):.1f}%)")

submission.head(10)

✓ Saved → T2_convnext_dann_uda_submission.csv  (11708 rows)
✓ Sanity checks passed

Predicted distribution:
  0 - Lateral_lying_left        18  (0.2%)
  1 - Lateral_lying_right       36  (0.3%)
  2 - Sitting                    0  (0.0%)
  3 - Standing               11559  (98.7%)
  4 - Sternal_lying             95  (0.8%)


,row_id,class_id
0,test_pen1_tur_cam1_20250920_174649_0000,3
1,test_pen1_tur_cam1_20250920_174649_0001,3
2,test_pen1_tur_cam1_20250920_174649_0002,3
3,test_pen1_tur_cam1_20250920_174649_0003,3
4,test_pen1_tur_cam1_20250920_174649_0004,3
5,test_pen1_tur_cam1_20250920_174649_0005,3
6,test_pen1_tur_cam1_20250920_174649_0006,3
7,test_pen1_tur_cam1_20250920_174649_0007,3
8,test_pen1_tur_cam1_20250920_174649_0008,3
9,test_pen1_tur_cam1_20250921_050022_0000,3
